# Cálculos de Poder Estadístico en Python
## Econometría Avanzada - Ana María Díaz

Este notebook replica los cálculos de poder estadístico vistos en clase.

In [ ]:
# Instalar dependencias si es necesario
!pip install statsmodels scipy numpy matplotlib -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.power import TTestIndPower, NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

## 1. Tamaño de muestra para un efecto dado

In [ ]:
# Parámetros
power = 0.8
alpha = 0.05
effect_size = 0.3  # Cohen's d

# Crear objeto de análisis de poder
analysis = TTestIndPower()

# Calcular tamaño de muestra
n_per_group = analysis.solve_power(effect_size=effect_size, 
                                    alpha=alpha, 
                                    power=power,
                                    ratio=1.0,
                                    alternative='two-sided')

print(f"Tamaño de muestra requerido por grupo: {np.ceil(n_per_group):.0f}")
print(f"Tamaño de muestra total: {np.ceil(n_per_group) * 2:.0f}")

## 2. Efecto mínimo detectable (MDE) para un N dado

In [ ]:
N_total = 1000
n_per_group = N_total / 2

# Calcular MDE
mde = analysis.solve_power(nobs1=n_per_group,
                           alpha=alpha,
                           power=power,
                           ratio=1.0,
                           alternative='two-sided')

print(f"Efecto mínimo detectable (Cohen's d): {mde:.4f}")

## 3. Curva de poder

In [ ]:
# Rango de tamaños de muestra
n_range = np.arange(50, 501, 10)

# Calcular poder para cada n
power_values = [analysis.power(effect_size=0.3, nobs1=n, alpha=0.05, ratio=1.0) 
                for n in n_range]

# Gráfico
plt.figure(figsize=(10, 6))
plt.plot(n_range, power_values, 'b-', linewidth=2, label='Poder')
plt.axhline(y=0.8, color='r', linestyle='--', label='Poder = 0.80')
plt.xlabel('Tamaño de muestra por grupo')
plt.ylabel('Poder estadístico')
plt.title('Curva de poder para d = 0.3')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 4. Comparación de proporciones

In [ ]:
from statsmodels.stats.power import zt_ind_solve_power

p1 = 0.08  # Proporción grupo control
p2 = 0.12  # Proporción grupo tratamiento

# Tamaño del efecto (h de Cohen)
h = proportion_effectsize(p2, p1)
print(f"Tamaño del efecto (h de Cohen): {h:.4f}")

# Calcular tamaño de muestra
n_prop = zt_ind_solve_power(effect_size=h, alpha=0.05, power=0.8, ratio=1.0)
print(f"Tamaño de muestra por grupo: {np.ceil(n_prop):.0f}")

## 5. Poder con covariables

In [ ]:
def power_with_covariates(n, effect, sd_original, r_squared_cov, alpha=0.05):
    """
    Calcula el poder estadístico incorporando la reducción de varianza por covariables.
    """
    # Desviación estándar residual
    sd_residual = sd_original * np.sqrt(1 - r_squared_cov)
    
    # Tamaño del efecto ajustado
    d_adjusted = effect / sd_residual
    
    # Calcular poder
    power = analysis.power(effect_size=d_adjusted, nobs1=n/2, alpha=alpha, ratio=1.0)
    return power

# Ejemplo
poder_sin_cov = analysis.power(effect_size=0.5/2, nobs1=100, alpha=0.05, ratio=1.0)
poder_con_cov = power_with_covariates(n=200, effect=0.5, sd_original=2, r_squared_cov=0.3)

print(f"Poder sin covariables (n=200): {poder_sin_cov:.4f}")
print(f"Poder con covariables (n=200, R²=0.3): {poder_con_cov:.4f}")

## 6. Diseño por conglomerados (clusters)

In [ ]:
def design_effect(icc, cluster_size):
    """Calcula el efecto de diseño para muestreo por conglomerados."""
    return 1 + icc * (cluster_size - 1)

# Parámetros
icc = 0.05  # Correlación intra-clase
m = 50      # Tamaño del cluster

de = design_effect(icc, m)
print(f"Efecto de diseño (ICC = {icc}, m = {m}): {de:.2f}")

# Tamaño de muestra simple
n_simple = np.ceil(analysis.solve_power(effect_size=0.3, alpha=0.05, power=0.8, ratio=1.0))

# Tamaño de muestra ajustado
n_cluster = np.ceil(n_simple * de)
k_clusters = np.ceil(n_cluster / m)

print(f"Tamaño de muestra simple por grupo: {n_simple:.0f}")
print(f"Tamaño de muestra ajustado por grupo: {n_cluster:.0f}")
print(f"Número de clusters necesarios por grupo: {k_clusters:.0f}")

## 7. Gráfico: Efecto del ICC en el tamaño de muestra

In [ ]:
icc_range = np.arange(0, 0.31, 0.01)
cluster_sizes = [20, 50, 100]

plt.figure(figsize=(10, 6))
for m in cluster_sizes:
    de_values = [design_effect(icc, m) for icc in icc_range]
    plt.plot(icc_range, de_values, label=f'm = {m}')

plt.xlabel('Correlación Intra-Clase (ICC)')
plt.ylabel('Efecto de Diseño')
plt.title('Efecto de diseño según ICC y tamaño de cluster')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()